# Branches Exploratory Data Analysis

## Purpose

This notebook checks whether branch records are complete, consistent and suitable for joining to organisations, programmes and downstream Pulse80 application data.

## Files used

- `Branches.csv` — branch definitions
- `Organisations.csv` — parent organisation definitions
- `Programmes.csv` — programmes assigned to organisations and branches

The current sample contains two branches belonging to one organisation. The checks confirm the current records, but a larger dataset will still be needed to judge behaviour across many organisations.

## 1. Prepare the notebook

The following block imports pandas and locates the repository's raw-data folder. It uses a relative search so the notebook works on different computers and does not contain or print anyone's personal file path.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

def find_raw_data_dir(start: Path = Path.cwd()) -> Path:
    for directory in (start, *start.parents):
        for candidate in (
            directory / "data" / "raw",
            directory / "data-analytics" / "data" / "raw",
        ):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError("Could not locate the repository raw-data folder.")

RAW_DATA_DIR = find_raw_data_dir()

## 2. Load the related datasets

This block loads the branch, organisation and programme files. The summary confirms how many rows and columns are available before any analysis is performed.

In [2]:
branches = pd.read_csv(RAW_DATA_DIR / "Branches.csv")
organisations = pd.read_csv(RAW_DATA_DIR / "Organisations.csv")
programmes = pd.read_csv(RAW_DATA_DIR / "Programmes.csv")

dataset_summary = pd.DataFrame({
    "dataset": ["Branches", "Organisations", "Programmes"],
    "rows": [len(branches), len(organisations), len(programmes)],
    "columns": [
        len(branches.columns),
        len(organisations.columns),
        len(programmes.columns),
    ],
})

dataset_summary

,dataset,rows,columns
0,Branches,2,6
1,Organisations,1,10
2,Programmes,1,11


## 3. Inspect the branch records

This block displays the branch records so their identifiers, names, cities, countries and creation dates can be reviewed directly.

In [3]:
branches

,branch_id,organisation_id,name,city,country,created_at
0,BR-001,ORG-001,Metsi Operations Site,Jwaneng,Botswana,2026-05-15T09:05:00Z
1,BR-002,ORG-001,Gaborone Head Office,Gaborone,Botswana,2026-05-15T09:06:00Z


### Explore the branch footprint

This block summarises how branches are distributed across organisations, cities and countries. It describes Pulse80's current operating footprint without treating a small count as a problem.

In [4]:
branches_per_organisation = (
    branches.groupby("organisation_id", dropna=False)
    .size()
    .rename("branch_count")
    .reset_index()
)

branches_per_city = (
    branches.groupby(["country", "city"], dropna=False)
    .size()
    .rename("branch_count")
    .reset_index()
)

branches_per_country = (
    branches.groupby("country", dropna=False)
    .size()
    .rename("branch_count")
    .reset_index()
)

print("Branches per organisation:")
display(branches_per_organisation)

print("Branches per city:")
display(branches_per_city)

print("Branches per country:")
display(branches_per_country)

Branches per organisation:


,organisation_id,branch_count
0,ORG-001,2


Branches per city:


,country,city,branch_count
0,Botswana,Gaborone,1
1,Botswana,Jwaneng,1


Branches per country:


,country,branch_count
0,Botswana,2


## 4. Profile completeness and uniqueness

This block summarises each column's data type, missing values and number of unique values. It also checks whether any complete rows have been duplicated.

In [5]:
branch_profile = pd.DataFrame({
    "data_type": branches.dtypes.astype(str),
    "missing_count": branches.isna().sum(),
    "unique_values": branches.nunique(dropna=False),
})

print("Fully duplicated rows:", branches.duplicated().sum())
branch_profile

Fully duplicated rows: 0


,data_type,missing_count,unique_values
branch_id,object,0,2
organisation_id,object,0,1
name,object,0,2
city,object,0,2
country,object,0,1
created_at,object,0,2


## 5. Validate branch identifiers and names

This block checks that every branch has an ID, each ID is unique, IDs follow the `BR-001` format, and the same organisation does not contain duplicate branch names after ignoring capitalisation and extra spaces.

In [6]:
normalised_branch_names = branches["name"].astype("string").str.strip().str.casefold()

identifier_checks = pd.Series({
    "missing_branch_ids": branches["branch_id"].isna().sum(),
    "duplicate_branch_ids": branches["branch_id"].duplicated().sum(),
    "invalid_branch_id_formats": (
        ~branches["branch_id"].astype("string").str.match(r"^BR-[0-9]{3,}$", na=False)
    ).sum(),
    "duplicate_names_within_organisation": (
        branches.assign(_normalised_name=normalised_branch_names)
        .duplicated(subset=["organisation_id", "_normalised_name"])
        .sum()
    ),
})

identifier_checks.to_frame("count")

,count
missing_branch_ids,0
duplicate_branch_ids,0
invalid_branch_id_formats,0
duplicate_names_within_organisation,0


### Confirm the identifier rules

These assertions stop the notebook if a required identifier rule fails. Passing them means branch IDs are present, unique, consistently formatted and safe to use as join keys.

In [7]:
assert identifier_checks.eq(0).all()
print("Branch identifier and name checks passed.")

Branch identifier and name checks passed.


## 6. Validate required text and creation dates

This block checks for blank branch names, cities, countries and organisation IDs. It also converts `created_at` into a real timestamp and detects invalid dates.

In [8]:
required_text_columns = ["organisation_id", "name", "city", "country"]

required_text_checks = pd.Series({
    column: (
        branches[column].isna()
        | branches[column].astype("string").str.strip().eq("").fillna(False)
    ).sum()
    for column in required_text_columns
})

branches["created_at"] = pd.to_datetime(
    branches["created_at"],
    utc=True,
    errors="coerce",
)

value_checks = pd.concat([
    required_text_checks.rename(lambda name: f"invalid_{name}"),
    pd.Series({"invalid_created_at_dates": branches["created_at"].isna().sum()}),
])

value_checks.to_frame("count")

,count
invalid_organisation_id,0
invalid_name,0
invalid_city,0
invalid_country,0
invalid_created_at_dates,0


### Confirm the required values

These assertions verify that required branch details are not blank and creation dates are readable. Passing them means the current records contain the basic information the application needs.

In [9]:
assert value_checks.eq(0).all()
print("Required branch value checks passed.")

Required branch value checks passed.


## 7. Validate the organisation relationship

Each branch must belong to an organisation that exists. This block joins branches to organisations, checks that no organisation reference is missing, and compares the branch country with the parent organisation's country.

In [10]:
branch_organisation_join = branches.merge(
    organisations[["organisation_id", "name", "country"]].rename(
        columns={
            "name": "organisation_name",
            "country": "organisation_country",
        }
    ),
    on="organisation_id",
    how="left",
    validate="many_to_one",
    indicator="organisation_join_status",
)

branch_organisation_join["country_matches"] = (
    branch_organisation_join["country"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq(
        branch_organisation_join["organisation_country"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )
    .fillna(False)
)

branch_organisation_join

,branch_id,organisation_id,name,city,country,created_at,organisation_name,organisation_country,organisation_join_status,country_matches
0,BR-001,ORG-001,Metsi Operations Site,Jwaneng,Botswana,2026-05-15 09:05:00+00:00,Kopano Mining Group,Botswana,both,True
1,BR-002,ORG-001,Gaborone Head Office,Gaborone,Botswana,2026-05-15 09:06:00+00:00,Kopano Mining Group,Botswana,both,True


### Confirm the organisation join

These assertions confirm that every branch found its parent organisation, the countries agree, and the join did not add or remove branch rows.

In [11]:
assert branch_organisation_join["organisation_join_status"].eq("both").all()
assert branch_organisation_join["country_matches"].all()
assert len(branch_organisation_join) == len(branches)
print("All branches join to their organisations correctly.")

All branches join to their organisations correctly.


## 8. Validate programme use of branches

Programmes refer to branches through `branch_id`. This block checks that every programme points to an existing branch and that the selected branch belongs to the same organisation as the programme.

In [12]:
programme_branch_join = programmes.merge(
    branches[["branch_id", "organisation_id", "name"]].rename(
        columns={
            "organisation_id": "branch_organisation_id",
            "name": "branch_name",
        }
    ),
    on="branch_id",
    how="left",
    validate="many_to_one",
    indicator="branch_join_status",
)

programme_branch_join["branch_belongs_to_programme_organisation"] = (
    programme_branch_join["organisation_id"]
    == programme_branch_join["branch_organisation_id"]
)

programme_branch_join

,programme_id,organisation_id,branch_id,name,programme_type,start_date,end_date,venue,status,target_participants,created_at,branch_organisation_id,branch_name,branch_join_status,branch_belongs_to_programme_organisation
0,PRG-001,ORG-001,BR-001,2026 Workforce Wellness Programme,screening,2026-06-18T00:00:00Z,2026-06-18T00:00:00Z,Metsi Operations Site - Wellness Hall,completed,28,2026-05-22T10:00:00Z,ORG-001,Metsi Operations Site,both,True


### Confirm the programme join

These assertions confirm that programme branch references exist, respect organisation ownership and preserve every programme row.

In [13]:
assert programme_branch_join["branch_join_status"].eq("both").all()
assert programme_branch_join["branch_belongs_to_programme_organisation"].all()
assert len(programme_branch_join) == len(programmes)
print("All programmes join to the correct branches.")

All programmes join to the correct branches.


## 9. Measure programme coverage by branch

This block counts programmes assigned to each branch. A branch with zero programmes is not necessarily an error; it may simply be available for future programme delivery.

In [14]:
programmes_per_branch = (
    programmes.groupby("branch_id")
    .size()
    .rename("programme_count")
)

branch_programme_coverage = (
    branches[["branch_id", "organisation_id", "name", "city", "country"]]
    .merge(
        programmes_per_branch,
        left_on="branch_id",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
)

branch_programme_coverage["programme_count"] = (
    branch_programme_coverage["programme_count"].fillna(0).astype(int)
)

branch_programme_coverage

,branch_id,organisation_id,name,city,country,programme_count
0,BR-001,ORG-001,Metsi Operations Site,Jwaneng,Botswana,1
1,BR-002,ORG-001,Gaborone Head Office,Gaborone,Botswana,0


### What the programme coverage means for Pulse80

The coverage table shows where programmes are currently being delivered and which registered branches remain available for future work. It can support branch selectors, organisation pages, programme planning and dashboard filters.

It does not show which branch has higher wellness risk or better participation. Those questions require a separate analysis joining branches to programmes, participations and screenings.

## 10. Summarise join readiness

This block brings the important validation results into one table. A value of zero for every failed check means the current branch data is suitable for downstream joins.

In [15]:
join_readiness = pd.Series({
    "fully_duplicated_rows": branches.duplicated().sum(),
    "missing_branch_ids": branches["branch_id"].isna().sum(),
    "duplicate_branch_ids": branches["branch_id"].duplicated().sum(),
    "invalid_branch_id_formats": identifier_checks["invalid_branch_id_formats"],
    "duplicate_names_within_organisation": identifier_checks[
        "duplicate_names_within_organisation"
    ],
    "invalid_required_values": required_text_checks.sum(),
    "invalid_created_at_dates": branches["created_at"].isna().sum(),
    "invalid_organisation_references": (
        ~branch_organisation_join["organisation_join_status"].eq("both")
    ).sum(),
    "branch_organisation_country_mismatches": (
        ~branch_organisation_join["country_matches"]
    ).sum(),
    "invalid_programme_branch_references": (
        ~programme_branch_join["branch_join_status"].eq("both")
    ).sum(),
    "programme_branch_organisation_mismatches": (
        ~programme_branch_join["branch_belongs_to_programme_organisation"]
    ).sum(),
    "branch_rows_lost_during_join": len(branches) - len(branch_organisation_join),
    "programme_rows_lost_during_join": len(programmes) - len(programme_branch_join),
})

join_readiness.to_frame("failed_records")

,failed_records
fully_duplicated_rows,0
missing_branch_ids,0
duplicate_branch_ids,0
invalid_branch_id_formats,0
duplicate_names_within_organisation,0
invalid_required_values,0
invalid_created_at_dates,0
invalid_organisation_references,0
branch_organisation_country_mismatches,0
invalid_programme_branch_references,0


### Final automated confirmation

This final assertion provides one clear pass or fail result for the current dataset. It fails immediately if any critical identifier or relationship problem remains.

In [16]:
assert join_readiness.eq(0).all()
print("All critical branch join-readiness checks passed.")

All critical branch join-readiness checks passed.


## 11. Simple conclusion

- All branch records are complete: names, cities, countries and creation dates are all filled in correctly.
- Every branch has a unique ID in the right format.
- All branches are correctly linked to a real organisation.
- Branch locations match their parent organisation's country.
- Every wellness programme is correctly linked to a real, existing branch.
- No data was lost or dropped when we combined the datasets.

What the data currently shows:
- We have two branches in Botswana under one organisation: one in Jwaneng, one in Gaborone. The Jwaneng branch has a wellness programme running and Gaborone doesn't yet.

What this data can be used for right now:
- Letting users select a branch
- Building organisation and branch pages
- Planning programmes
- Filtering dashboards
- Reporting on programmes by branch

What it can't tell us yet:
This information can't yet tell us which branch has higher health risk, stronger employee participation or greater need for wellness support. To answer that, we'd need to connect this branch data with programme participation and screening results.

Bottom line: The branch data file is complete, accurate, and ready to be connected to organisations, programmes, and the rest of the app.